In [2]:
# In thông tin sinh viên theo yêu cầu
print("Họ và tên: Vũ Trương Phi")
print("MSSV: 6451071059")
print("-" * 40)

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

# Bước 1: Đọc dữ liệu Online Retail nếu chưa có sẵn trong bộ nhớ
if 'df' not in locals():
    df = pd.read_excel('Online Retail.xlsx')
    # Tiền xử lý cơ bản: làm sạch Description, bỏ giá trị khuyết và hóa đơn hủy (chứa 'C')
    df['Description'] = df['Description'].str.strip()
    df.dropna(axis=0, subset=['InvoiceNo'], inplace=True)
    df['InvoiceNo'] = df['InvoiceNo'].astype('str')
    df = df[~df['InvoiceNo'].str.contains('C')]

# Bước 2: Lọc dữ liệu hóa đơn của Nước Đức ('Germany') và gom nhóm
basket_germany = (df[df['Country'] == "Germany"]
                  .groupby(['InvoiceNo', 'Description'])['Quantity']
                  .sum().unstack().reset_index().fillna(0)
                  .set_index('InvoiceNo'))

# Bước 3: Chuyển dữ liệu sang dạng One-Hot Encoding (0 và 1)
def encode_units(x):
    if x <= 0:
        return 0
    if x >= 1:
        return 1

basket_germany = basket_germany.applymap(encode_units)

# Bước 4: Xóa cột cước phí 'POSTAGE' nếu tồn tại
if 'POSTAGE' in basket_germany.columns:
    basket_germany.drop('POSTAGE', inplace=True, axis=1)

# Bước 5: Tìm tập phổ biến bằng Apriori với min_sup = 5% (0.05)
itemsets_germany = apriori(basket_germany, min_support=0.05, use_colnames=True)
print("10 tập mục phổ biến đầu tiên tìm được:")
display(itemsets_germany.head(10))
print("-" * 40)

# Bước 6: Tạo tập luật kết hợp với min_conf = 50% (0.5)
rules_germany = association_rules(itemsets_germany, metric="confidence", min_threshold=0.5)

# Chuyển đổi frozenset sang chuỗi unicode để hiển thị
rules_germany["antecedents"] = rules_germany["antecedents"].apply(lambda x: list(x)[0]).astype("unicode")
rules_germany["consequents"] = rules_germany["consequents"].apply(lambda x: list(x)[0]).astype("unicode")

print(f"Tổng số luật kết hợp tìm được: {len(rules_germany)}")
print("Danh sách các luật kết hợp:")
for i in range(len(rules_germany)):
    print(rules_germany.loc[i, 'antecedents'], '==>', rules_germany.loc[i, 'consequents'],
          '[ support:', round(rules_germany.loc[i, 'support'], 4),
          ', confidence:', round(rules_germany.loc[i, 'confidence'], 4), ']')

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Họ và tên: Vũ Trương Phi
MSSV: 6451071059
----------------------------------------
10 tập mục phổ biến đầu tiên tìm được:


,support,itemsets
0,0.102845,(6 RIBBONS RUSTIC CHARM)
1,0.070022,(ALARM CLOCK BAKELIKE PINK)
2,0.065646,(CHARLOTTE BAG APPLES DESIGN)
3,0.050328,(CHILDRENS CUTLERY DOLLY GIRL)
4,0.061269,(COFFEE MUG APPLES DESIGN)
5,0.063457,(FAWN BLUE HOT WATER BOTTLE)
6,0.072210,(GUMBALL COAT RACK)
7,0.056893,(IVORY KITCHEN SCALES)
8,0.063457,(JAM JAR WITH PINK LID)
9,0.091904,(JAM MAKING SET PRINTED)


----------------------------------------
Tổng số luật kết hợp tìm được: 8
Danh sách các luật kết hợp:
PLASTERS IN TIN CIRCUS PARADE ==> PLASTERS IN TIN WOODLAND ANIMALS [ support: 0.0678 , confidence: 0.5849 ]
PLASTERS IN TIN SPACEBOY ==> PLASTERS IN TIN WOODLAND ANIMALS [ support: 0.0613 , confidence: 0.5714 ]
PLASTERS IN TIN WOODLAND ANIMALS ==> ROUND SNACK BOXES SET OF4 WOODLAND [ support: 0.0744 , confidence: 0.5397 ]
RED RETROSPOT CHARLOTTE BAG ==> WOODLAND CHARLOTTE BAG [ support: 0.0591 , confidence: 0.8438 ]
ROUND SNACK BOXES SET OF 4 FRUITS ==> ROUND SNACK BOXES SET OF4 WOODLAND [ support: 0.1313 , confidence: 0.8333 ]
ROUND SNACK BOXES SET OF4 WOODLAND ==> ROUND SNACK BOXES SET OF 4 FRUITS [ support: 0.1313 , confidence: 0.5357 ]
SPACEBOY LUNCH BOX ==> ROUND SNACK BOXES SET OF4 WOODLAND [ support: 0.07 , confidence: 0.6809 ]
WOODLAND CHARLOTTE BAG ==> ROUND SNACK BOXES SET OF4 WOODLAND [ support: 0.0635 , confidence: 0.5 ]
